---
title: "Partitioned Conjugate-Heat-Transfer Coupling"
date: 2026-10-13
categories: [multiphysics, conjugate heat transfer, coupling algorithms]
description: "Compare fixed relaxation and Aitken acceleration in a distributed thermal interface iteration."
---

# Partitioned Conjugate-Heat-Transfer Coupling
### When two individually stable solvers make a slow coupled solver

**Learning goal:** Build a partitioned conjugate-heat-transfer solver, diagnose its interface fixed-point convergence, and compare unrelaxed, fixed-relaxation, and Aitken-accelerated coupling against a monolithic reference.

**Time:** 60-90 minutes. Python, NumPy, SciPy, Matplotlib. Use one fixed spatial grid with `N = 80`; this is not a grid-convergence exercise.

**Engineering connection:** This is the algorithmic core of coupling separate CFD and thermal/system solvers: define exchanged data, enforce interface consistency, monitor a coupling residual, and accelerate the outer iteration without hiding failed physics solves.

Implementation cells intentionally contain TODOs only. Do not tune relaxation by looking only at the final temperature plot; convergence history and conservation must agree.

## 1. Coupled reduced model

Use a one-dimensional streamwise coordinate $x\in[0,1]$ with two fields on the same fixed nodes:

$$C_f\frac{dT_f}{dx}=H(T_s-T_f),\qquad T_f(0)=T_{in},$$
$$-K_s\frac{d^2T_s}{dx^2}+H(T_s-T_f)=q_v(x),\qquad T_s'(0)=T_s'(1)=0.$$

$T_f$ is a plug-flow coolant temperature and $T_s$ is the axial solid temperature. $H(T_s-T_f)$ is the distributed interfacial heat transfer. Use backward upwind differencing for the fluid equation and conservative second differences for the solid equation.

Write the discrete block system as

$$A_fT_f+C_{fs}T_s=b_f,$$
$$C_{sf}T_f+A_sT_s=b_s.$$

Set `N=80`, $C_f=1$, $K_s=0.02$, $H=100$, $T_{in}=0$, and
$$q_v(x)=8\exp[-((x-0.72)/0.12)^2]+1.$$

Use second-order one-sided Neumann rows or a finite-volume boundary treatment. Document the choice and use it identically in the partitioned and monolithic systems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import bmat, csc_matrix, diags
from scipy.sparse.linalg import splu, spsolve

N = 80
CF = 1.0
KS = 0.02
H_VALUES = [5.0, 25.0, 100.0, 500.0]
T_IN = 0.0
TOL = 1.0e-10
MAXITER = 500

# TODO: Set deterministic plotting/printing options and record package versions.


## 2. Assemble two reusable physics solvers

Implement `build_operators(H)` returning the four block matrices and right-hand sides. Factor $A_f$ and $A_s$ once for each value of $H$.

Implement:

- `solve_fluid(Ts, problem)`: solve $A_fT_f=b_f-C_{fs}T_s$.
- `solve_solid(Tf, problem)`: solve $A_sT_s=b_s-C_{sf}T_f$.
- `solve_monolithic(problem)`: solve the $2N$-by-$2N$ block system once as an independent coupled reference.
- `block_residual(Tf, Ts, problem)`: return normalized fluid, solid, and combined residual norms.

Do not form matrix inverses. Check matrix shapes, sparsity, finite entries, and boundary rows before iterating.

In [ ]:
def build_operators(H, n=N):
    """Assemble and factor the fixed-grid fluid and solid blocks."""
    # TODO: Return x, dx, Af, Cfs, Csf, As, bf, bs, and LU factors.
    raise NotImplementedError

def solve_fluid(Ts, problem):
    # TODO: One fluid subsolve.
    raise NotImplementedError

def solve_solid(Tf, problem):
    # TODO: One solid subsolve.
    raise NotImplementedError

def solve_monolithic(problem):
    # TODO: Solve the assembled 2N by 2N block system.
    raise NotImplementedError

def block_residual(Tf, Ts, problem):
    # TODO: Return normalized fluid, solid, and combined residual norms.
    raise NotImplementedError


## 3. Dirichlet-Neumann fixed-point coupling

Given a solid iterate $T_s^k$, perform a block Gauss-Seidel sweep:

$$T_f^{k+1}=A_f^{-1}(b_f-C_{fs}T_s^k),$$
$$\widehat T_s^{k+1}=A_s^{-1}(b_s-C_{sf}T_f^{k+1}).$$

Define the interface/coupling residual
$$r^k=\widehat T_s^{k+1}-T_s^k,$$
and update
$$T_s^{k+1}=T_s^k+\omega_k r^k.$$

Implement one driver that accepts a relaxation policy and records iteration, $\|r^k\|_2/\sqrt N$, combined block residual, $\omega_k$, and subsolve counts. Stop only when both coupling and block residuals are below tolerance. Treat reaching `MAXITER` as a failure, not a converged result.

In [ ]:
def partitioned_solve(problem, policy, omega0=1.0, tol=TOL, maxiter=MAXITER):
    """Run strong partitioned coupling and return state plus full history."""
    # TODO: Implement the Gauss-Seidel exchange and robust stopping logic.
    raise NotImplementedError


## 4. Fixed relaxation and vector Aitken acceleration

Compare three policies:

1. Unrelaxed: $\omega_k=1$.
2. Fixed: sweep $\omega\in\{0.2,0.5,0.8,1.0,1.2,1.5\}$.
3. Vector Aitken delta-squared, using consecutive residuals:

$$\omega_k=-\omega_{k-1}\frac{(r^{k-1})^T(r^k-r^{k-1})}{\|r^k-r^{k-1}\|_2^2}.$$

Use $\omega_0=1$ and clip the dynamic value to `[0.05, 1.5]`. If the denominator is too small or nonfinite, retain the previous valid value. Record every accepted relaxation factor.

Important: form $r^k$ from the **unrelaxed** solid candidate. Applying Aitken to already relaxed residuals changes the method.

In [ ]:
def fixed_policy(omega):
    # TODO: Return a callable compatible with partitioned_solve.
    raise NotImplementedError

def aitken_policy(omega0=1.0, bounds=(0.05, 1.5)):
    # TODO: Return a stateful callable implementing vector Aitken relaxation.
    raise NotImplementedError


## 5. Coupling-strength experiment

For each $H\in\{5,25,100,500\}$:

- Compute the monolithic reference.
- Run unrelaxed coupling, every fixed relaxation value, and Aitken acceleration.
- Record convergence status, iterations, final coupling residual, final block residual, and relative error against the monolithic stacked state.
- Estimate the asymptotic contraction factor from a linear fit to `log(coupling residual)` over a clean late-iteration window. Reject windows that include zeros, stagnation, or fewer than five points.

Do not change `N` between cases. The independent variable is coupling strength, not grid spacing.

In [ ]:
# TODO: Run the full method-by-H matrix and collect a tidy results table.
# Keep failed/nonconverged runs in the table rather than silently dropping them.


## 6. Required diagnostics and interpretation

Create:

1. $T_f(x)$ and $T_s(x)$ for the strongest converged case, overlaying the monolithic reference.
2. Coupling residual versus iteration for all policies at $H=100$ on a semilog axis.
3. Iteration count versus $H$ for each policy; mark failures distinctly.
4. Aitken $\omega_k$ versus iteration for $H=100$.
5. A compact table of contraction estimates, subsolve counts, final block residuals, and monolithic errors.

Answer: Why can both subsolvers be stable while the coupled iteration is slow or unstable? When does over-relaxation help? What does Aitken infer from residual history? Why is the monolithic solve a valuable algorithmic reference but not necessarily the preferred production architecture?

In [ ]:
# TODO: Generate all plots, the summary table, and concise interpretation.


## 7. Verification and success criteria

Assert all of the following:

- Every assembled block has the expected shape and finite entries; factorizations are reused.
- The monolithic combined residual is below $10^{-11}$ for every $H$.
- Every run labeled converged has coupling and combined block residuals below $10^{-10}$.
- Every converged partitioned state agrees with the monolithic state to relative 2-norm error below $10^{-8}$.
- Subsolve accounting is exact: one fluid and one solid solve per coupling sweep.
- Aitken relaxation values are finite and remain inside the clip interval.
- At $H=100$, Aitken uses fewer iterations than unrelaxed coupling. If it does not, diagnose residual definition, update order, and the first Aitken step before changing parameters.
- A discrete global heat balance is reported. With insulated solid ends, total generated heat must equal coolant enthalpy rise to the tolerance implied by your boundary discretization.

Verification is performed on one fixed grid and checks algebra, coupling consistency, and conservation - not observed spatial order.

In [ ]:
# TODO: Implement automatic checks tied to the actual results.


## 8. Stretch goal and references

**Stretch goal (15-25 minutes):** Replace scalar Aitken relaxation with Anderson acceleration of depth 3 for the solid interface vector. Compare iteration count and stored-vector cost at $H=500$. Keep the same physics matrices and grid. Alternatively, reverse the exchange order and explain the change in convergence.

### References

- **Dedicated project reference:** Victor Van Riet, Wim Beyne, Michel De Paepe, and Joris Degroote, "Convergence Behaviour of Partitioned Methods for Conjugate Heat Transfer Problems," *ECCOMAS Congress 2024*, CIMNE, 2024. DOI: 10.23967/eccomas.2024.127. [Open paper and metadata](https://biblio.ugent.be/publication/01JDHA0A458RSWG0HVT42TH90J). Focus on interface variables, Biot/coupling-strength effects, fixed relaxation, Aitken acceleration, and residual histories.
- **Paper to read:** Sarah Nataj, David C. Del Rey Fernandez, David Brown, et al., "A Finite-Difference Summation-by-Parts Conditionally Stable Partitioned Algorithm for Conjugate Heat Transfer Problems," *Journal of Scientific Computing* 109, 53 (2026). DOI: 10.1007/s10915-026-03462-x. [Open-access article](https://link.springer.com/article/10.1007/s10915-026-03462-x).

### Topic ledger

| Date / record | Primary objective | Status |
|---|---|---|
| Older assignments, dates unavailable | Grid convergence and errors | User reported five repetitive assignments; individual records unavailable |
| 2026-10-05 | Grid convergence and discretization uncertainty | Inspected |
| 2026-10-06 corrected | Fixed-mesh advection: CFL, conservation, positivity, TV | Inspected |
| 2026-10-07 | Krylov methods and line-block preconditioning | Inspected |
| 2026-10-08 | Data layout, cache behavior, and flux-kernel throughput | Inspected |
| 2026-10-09 | Nonlinear natural-circulation thermal-fluid modeling | Inspected |
| 2026-10-12 | Discrete-adjoint PDE-constrained optimization | Inspected |
| 2026-10-13 | Partitioned multiphysics coupling and relaxation | This assignment |

The six immediately preceding distinct objectives are recoverable; the seventh older record is only available through the user's report. This is assignment 5 of the ten-assignment exclusion established October 6; five excluded assignments remain after today. Planned next objectives: high-order reconstruction, parallel algorithms, time integration, uncertainty propagation, then turbulence modeling. No grid-refinement, observed-order, Richardson, or GCI task appears here.